# GeoDiff-TrustMoE: residual recovery on OLI2MSI

Your prior test table measured base PSNR 34.159966 dB and residual gains below
0.00071 dB. This notebook checks the trained weights, tests residual learning,
then compares HR-conditioned experts and routing that predicts correction benefit.

**Protocol stays 128 x 128 actual Landsat LR -> 384 x 384 Sentinel HR.**
Training draws aligned random crops from the 160->480 source frames. Validation
uses 255 hash-selected training sources; test uses all 100 official test centers
when validation_percent=5 and seed=42. The fixed clip(.0,.3)/.3 radiometry and
optional uint8 grid match the previous notebook. These are center-crop results,
not the paper's 32->96 results or full-frame 160->480 results.

Train a useful single expert before comparing MoE. Experts see LR features plus
signed HR base detail, with actual sparse tile dispatch. The gain router learns
the observed fractional MSE reduction of a proposal. HR provides training labels
only. There is still one frozen base pass, no diffusion and no PixelShuffle.

Run top to bottom. Controls are epochs and batch size. Training resumes at the
last completed epoch; completed evaluations are reused. Use a new suite for this
changed objective. Final cells save exact outputs, comparisons and a download ZIP.
Improvement and research novelty must be established by the resulting experiments.


## 1. Controls: official data, 128-pixel input, epochs and batch size


In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, hashlib, time, zipfile

FAST_DEV_RUN = False
REPOSITORY_URL = 'https://github.com/shashankjs2002/SI-SR-1.git'
REPOSITORY_BRANCH = '3x-continued'
REPOSITORY_DIR = Path('/kaggle/working/geodiff-trust-recovery-source')
SUITE_ROOT = Path('/kaggle/working/geodiff-trust-recovery-oli128-v2')
KNOWN_DATA_ROOT = Path('/kaggle/input/datasets/twilight2002/oli2msi-thesis/OLI2MSI-dataset/OLI2MSI')
OLI2MSI_DATA_ROOT = KNOWN_DATA_ROOT if KNOWN_DATA_ROOT.exists() else None
RESTORE_SUITE_FROM = None
OLD_SUITE_ROOT = Path('/kaggle/working/geodiff-trust-moe-oli2msi-128-v1')
# Set OLD_SUITE_ROOT to your attached previous outputs if this is a fresh session.
BASE_CHECKPOINTS = {42: None}  # Optional exact best.pt paths, one per requested seed.
IMPORT_BASE_CHECKPOINT = None

EPOCHS = {'base': 20, 'diagnostic': 60, 'single': 15, 'residual': 10}
BATCH_SIZE = 4
TRAIN_LR_CROP = 128
NUM_EXPERTS, TOP_K = 5, 2
REGION_FRACTION = 0.5
SEEDS = [42]
RUN = dict(dense_gain=True, sparse_error=True, sparse_gain=True,
           sparse_no_trust=True, sparse_uniform=True, sparse_conv=False,
           sparse_adversarial=False, adaptive_k=False, legacy_expert_control=False)
RUN_TEST_EVALUATION = False  # Set True only after reviewing the validation cells.
DIAGNOSTIC_PAIRS = 8
REQUIRE_RECOVERY_SCREEN = True
BASE_MODEL = dict(base_embed_dim=32, base_depth=2, base_groups=2, base_heads=4,
                  window_size=8, width=32, tile_size=8, halo=4)
DISPLAY_MAX = 1.0
DATASET_PROTOCOL = 'provided_benchmark'
VALIDATION_PERCENT = 5
QUANTIZE_TO_UINT8_GRID = True
FAST_LIMIT = 16 if FAST_DEV_RUN else None

if sys.version_info < (3, 10):
    raise RuntimeError('Use the current Python 3.10+ Kaggle kernel.')
if not 1 <= TOP_K <= NUM_EXPERTS or not 0 < REGION_FRACTION <= 1:
    raise ValueError('Invalid expert count/top-k/region fraction')
SUITE_ROOT.mkdir(parents=True, exist_ok=True)

def run(command, cwd=None):
    environment = os.environ.copy()
    environment['PYTHONPATH'] = str(REPOSITORY_DIR / 'src') + os.pathsep + environment.get('PYTHONPATH', '')
    environment['PYTHONUNBUFFERED'] = '1'
    print('+', ' '.join(map(str, command)), flush=True)
    return subprocess.run(list(map(str, command)), cwd=cwd, env=environment, check=True)

print('Suite:', SUITE_ROOT)
print('Model geometry: 128 x 128 -> 384 x 384')
print('Epochs:', EPOCHS, 'batch size:', BATCH_SIZE, 'experts:', NUM_EXPERTS, 'top-k:', TOP_K)


## 2. Clone the exact branch, install it, restore outputs and verify GPU


In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', 'rasterio', 'numpy', 'Pillow',
     'PyYAML', 'tqdm', 'pandas', 'matplotlib'])
if not REPOSITORY_DIR.exists():
    run(['git', 'clone', '--depth', '1', '--single-branch', '--branch',
         REPOSITORY_BRANCH, REPOSITORY_URL, REPOSITORY_DIR])
if not (REPOSITORY_DIR / '.git').is_dir():
    raise RuntimeError('REPOSITORY_DIR exists but is not a Git checkout; use another path.')
branch = subprocess.check_output(['git', '-C', str(REPOSITORY_DIR), 'branch', '--show-current'], text=True).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f'Expected {REPOSITORY_BRANCH}, found {branch}; use another repository path.')
required = REPOSITORY_DIR / 'src/geodiff_gan/experiments/trust_recovery.py'
if not required.is_file():
    raise RuntimeError('Push the OLI2MSI TrustMoE notebook support to GitHub, then clone into a new path.')
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPOSITORY_DIR, '--no-deps'])
sys.path.insert(0, str(REPOSITORY_DIR / 'src'))

if RESTORE_SUITE_FROM:
    source = Path(RESTORE_SUITE_FROM)
    if source.is_file():
        expanded = SUITE_ROOT.parent / 'trust-moe-oli2msi-restore'
        expanded.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(source) as archive:
            for member in archive.infolist():
                if not (expanded / member.filename).resolve().is_relative_to(expanded.resolve()):
                    raise ValueError('Unsafe path in restore ZIP: ' + member.filename)
                if (member.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError('Restore ZIP symlinks are not accepted')
            archive.extractall(expanded)
        source = expanded
    if not source.is_dir():
        raise FileNotFoundError('RESTORE_SUITE_FROM must be a prior result ZIP or extracted suite directory.')
    for path in source.rglob('*'):
        if path.is_file():
            destination = SUITE_ROOT / path.relative_to(source)
            if not destination.exists():
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, destination)

import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, FileLink
from geodiff_gan.data.oli2msi_128 import find_oli2msi_layout, prepare_oli2msi_128
from geodiff_gan.experiments.trust_moe import (
    make_config, prepare_manifest, write_json, digest_file, adopt_base, evaluate, dataset_for,
)
from geodiff_gan.experiments.trust_report import (
    METRICS, summarize, benchmark, compare_controls, paired_intervals, visualize, bundle_results,
)
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator.')
print('Commit:', subprocess.check_output(['git', '-C', str(REPOSITORY_DIR), 'rev-parse', 'HEAD'], text=True).strip())
print('Python:', sys.version, 'Torch:', torch.__version__, 'GPU:', torch.cuda.get_device_name(0))
write_json(SUITE_ROOT / 'environment.json', dict(python=sys.version, torch=torch.__version__,
    cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0), input=[128, 128], output=[384, 384]))

from geodiff_gan.experiments.trust_moe import load_model
from geodiff_gan.experiments.trust_recovery import (
    RECOVERY_VERSION, recovery_config, audit_checkpoint, memorization_report, validation_screen,
)
assert RECOVERY_VERSION == 'trust-recovery-v2', 'Clone the updated 3x-continued branch.'
for name in ('figures', 'reports', 'diagnostics', 'source_receipts'):
    (SUITE_ROOT / name).mkdir(exist_ok=True)
write_json(SUITE_ROOT / 'source_receipts/revision.json', {
    'commit': subprocess.check_output(['git', '-C', str(REPOSITORY_DIR), 'rev-parse', 'HEAD'], text=True).strip(),
    'recovery_version': RECOVERY_VERSION,
})
if FAST_DEV_RUN:
    EPOCHS = {'base': 1, 'diagnostic': 2, 'single': 1, 'residual': 3}
    REQUIRE_RECOVERY_SCREEN = False
    RUN_TEST_EVALUATION = False
    print('SMOKE RUN: exercises plumbing; not a performance experiment.')


## 3. Discover and prepare official OLI2MSI pairs

Attach a Kaggle dataset containing `train_lr`, `train_hr`, `test_lr`, and
`test_hr`. `OLI2MSI_DATA_ROOT` may point to the dataset root, one split folder,
or an individual TIFF. A full run requires exactly 5,225 official training and
100 official test pairs. Missing/corrupt pairs stop the benchmark rather than
silently reducing it.

Prepared arrays are an internal restart cache for the current loader. Training
sources remain full 160→480 so every epoch can draw a different aligned 128→384
crop. Validation/test arrays are fixed center crops. The original attached TIFFs
are read-only and never changed or deleted.


In [ ]:
if OLI2MSI_DATA_ROOT is None:
    search = []
    for directory in Path('/kaggle/input').rglob('*'):
        if directory.is_dir() and ''.join(c for c in directory.name.casefold() if c.isalnum()) == 'trainlr':
            search.append(directory)
    roots = sorted({path.parent for path in search})
    if len(roots) != 1:
        raise ValueError(f'Set OLI2MSI_DATA_ROOT explicitly; candidate roots: {roots}')
    OLI2MSI_DATA_ROOT = roots[0]
PREPARED_MANIFEST, DATA_CARD = prepare_oli2msi_128(
    OLI2MSI_DATA_ROOT, SUITE_ROOT / 'prepared_oli2msi', seed=42,
    validation_percent=VALIDATION_PERCENT, quantize=QUANTIZE_TO_UINT8_GRID,
    fast_limit=FAST_LIMIT, require_official_counts=not FAST_DEV_RUN)
MANIFEST = SUITE_ROOT / 'runtime_manifest.jsonl'
AUDIT = prepare_manifest(PREPARED_MANIFEST, MANIFEST, spatial_audit=False,
    minimum_test_fraction=0, disjoint_tiles=False)
AUDIT['oli2msi_protocol'] = DATA_CARD
write_json(SUITE_ROOT / 'dataset_audit.json', AUDIT)
print(json.dumps(AUDIT, indent=2))
if AUDIT['frames_hr'] != [384, 480]:
    raise RuntimeError(f'Expected train/eval HR frame sizes [384, 480], got {AUDIT["frames_hr"]}')


## 4. Verify the exact 128 → 384 tensors before training


In [ ]:
from torch.nn import functional as F

def preview_pair(index=0, split='val'):
    config = make_config(MANIFEST, SUITE_ROOT, profile='base', crop_size=TRAIN_LR_CROP)
    data = dataset_for(config, split)
    sample = data[int(index) % len(data)]
    lr, hr = sample['lr'], sample['hr']
    if tuple(lr.shape) != (3, 128, 128) or tuple(hr.shape) != (3, 384, 384):
        raise RuntimeError(f'Wrong model geometry: {tuple(lr.shape)} -> {tuple(hr.shape)}')
    bicubic = F.interpolate(lr[None], size=hr.shape[-2:], mode='bicubic', align_corners=False)[0].clamp(0, 1)
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    for axis, image, title in zip(axes, (lr, bicubic, hr),
            ('Landsat OLI LR 128 x 128', 'Bicubic 384 x 384', 'Sentinel-2 MSI HR 384 x 384')):
        axis.imshow(image.permute(1, 2, 0).clamp(0, 1), interpolation='nearest')
        axis.set_title(title); axis.axis('off')
    fig.tight_layout()
    folder = SUITE_ROOT / 'figures'; folder.mkdir(exist_ok=True)
    fig.savefig(folder / f'pair_{split}_{index}.png', dpi=160)
    plt.show()
    print('Stored ranges:', float(lr.min()), float(lr.max()), float(hr.min()), float(hr.max()))
preview_pair(0, 'val')


## 5. Reuse your trained base and audit the old residual

The imported base is copied into this suite. All comparisons use exactly those
frozen weights. If no base can be found, this cell trains one. Attach your prior
Kaggle outputs and set OLD_SUITE_ROOT to avoid repeating base training.
Epoch increases resume; architecture/loss changes need a new experiment directory.


In [ ]:
BASES, SINGLES, MODELS, CONFIGS = {}, {}, {}, {}

def launch(config):
    root = Path(config['root']); root.mkdir(parents=True, exist_ok=True)
    request = root / 'request_config.json'
    write_json(request, config)
    run([sys.executable, '-m', 'geodiff_gan.cli.trust_moe', 'train', '--config', request], cwd=REPOSITORY_DIR)
    checkpoint = root / 'best.pt'
    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)
    return checkpoint

def read_checkpoint_config(checkpoint):
    return torch.load(checkpoint, map_location='cpu', weights_only=False)['config']

for seed in SEEDS:
    config = make_config(MANIFEST, SUITE_ROOT / f'seed_{seed}/base', profile='base',
        seed=seed, epochs=EPOCHS['base'], batch_size=BATCH_SIZE, crop_size=TRAIN_LR_CROP,
        experts=NUM_EXPERTS, top_k=TOP_K, dataset_id=AUDIT['dataset_id'], base_model=BASE_MODEL)
    config['training']['progress'] = 'compact'
    requested = BASE_CHECKPOINTS.get(seed)
    old = Path(requested) if requested else OLD_SUITE_ROOT / f'seed_{seed}/base/best.pt'
    if old.is_file():
        prior = read_checkpoint_config(old)
        if prior['dataset_id'] != AUDIT['dataset_id']:
            raise ValueError('Old base dataset differs. Check data protocol and manifest before importing.')
        # Honor the real base dimensions recorded by the trained checkpoint.
        for key in ('base_embed_dim', 'base_depth', 'base_groups', 'base_heads', 'window_size'):
            if key in prior['model']:
                config['model'][key] = prior['model'][key]
        BASES[seed] = adopt_base(config, old)
        print('Imported trained base:', old)
    else:
        print('No imported base found; training the base for this seed.')
        BASES[seed] = launch(config)
    MODELS[f'{seed}/base'] = BASES[seed]
    CONFIGS[f'{seed}/base'] = config
    write_json(SUITE_ROOT / f'seed_{seed}/base/lineage.json', {
        'source': str(old) if old.is_file() else 'trained in this suite',
        'source_sha256': digest_file(old) if old.is_file() else None,
        'checkpoint_sha256': digest_file(BASES[seed]),
    })
    old_expert = OLD_SUITE_ROOT / f'seed_{seed}/single_expert/best.pt'
    if old_expert.exists():
        previous = audit_checkpoint(old_expert, MANIFEST, SUITE_ROOT / f'diagnostics/old_single_{seed}.json')
        display(pd.DataFrame(previous['rows']))
        print('Reconstruction gradients:', previous.get('reconstruction_gradients'))
        print('Full objective gradients:', previous.get('full_loss_gradients'))
write_json(SUITE_ROOT / 'base_registry.json', {str(k): str(v.relative_to(SUITE_ROOT)) for k, v in BASES.items()})


## 6. Eight-image training diagnostic: can an expert learn a correction?

This uses fixed TRAIN crops, full coverage, one expert, trust off and guard off.
Its PSNR is memorization performance, never a validation/test score. It is not
used to initialize the scientific runs. A failed screen stops the full study so
you can share the diagnostic ZIP. No target is passed into the inference model.


In [ ]:
def new_config(seed, label, profile='single_expert', initializer=None, risk='gain', hr=True):
    dimensions = {k: v for k, v in CONFIGS[f'{seed}/base']['model'].items()
                  if k.startswith('base_') or k == 'window_size'}
    dimensions.update(width=BASE_MODEL['width'], tile_size=BASE_MODEL['tile_size'])
    return recovery_config(MANIFEST, SUITE_ROOT / f'seed_{seed}/{label}', parent=BASES[seed],
        initializer=initializer, profile=profile, seed=seed,
        epochs=EPOCHS['single'] if profile == 'single_expert' else EPOCHS['residual'],
        batch_size=BATCH_SIZE, crop_size=TRAIN_LR_CROP, experts=NUM_EXPERTS,
        top_k=TOP_K, dataset_id=AUDIT['dataset_id'], base_model=dimensions,
        risk_target=risk, coverage=REGION_FRACTION, hr_conditioning=hr)

DIAGNOSTICS = []
for seed in SEEDS:
    diagnostic = new_config(seed, 'diagnostic_overfit')
    diagnostic['training'].update(diagnostic_overfit_pairs=DIAGNOSTIC_PAIRS,
        epochs=EPOCHS['diagnostic'], batch_size=1, num_workers=0, learning_rate=1e-3,
        lr_epoch_decay=1.0)
    path = launch(diagnostic)
    report = memorization_report(path, SUITE_ROOT / f'diagnostics/memorization_{seed}.json')
    DIAGNOSTICS.append(report)
    display(pd.DataFrame(report['rows']))
    print('TRAIN-CROP diagnostic:', report['mean_psnr_gain'], 'dB; passed:', report['passed'])
if REQUIRE_RECOVERY_SCREEN and not all(r['passed'] for r in DIAGNOSTICS):
    archive = SUITE_ROOT.parent / (SUITE_ROOT.name + '_diagnostic.zip')
    bundle_results(SUITE_ROOT, archive, source_root=REPOSITORY_DIR)
    display(FileLink(str(archive.relative_to(Path('/kaggle/working')))))
    raise RuntimeError('Expert did not pass the train-crop learning check. Share diagnostics before running the full suite.')


## 7. Full training: one HR-conditioned residual expert

This trains on all 4,970 designated training sources, using independent fresh
residual initialization and the common frozen base. Its checkpoint is selected
on the 255 validation sources. This is the initializer for every subsequent MoE.


In [ ]:
for seed in SEEDS:
    config = new_config(seed, 'single_recovery')
    CONFIGS[f'{seed}/single_recovery'] = config
    SINGLES[seed] = launch(config)
    MODELS[f'{seed}/single_recovery'] = SINGLES[seed]


## 8. Validate the single expert before training all routers

Development screen: mean validation gain at least +0.01 dB, and over half the
images improve. This detects useful learning; it is not a publication criterion.
The report compares raw/EMA behavior and logs individual-module gradient norms.


In [ ]:
VAL_EVALS, SINGLE_SCREENS = {}, []

def eval_path(name, split, checkpoint, suffix='trained'):
    return SUITE_ROOT / 'evaluation' / split / name / (digest_file(checkpoint)[:12] + '_' + suffix)

for seed in SEEDS:
    for label, checkpoint in (('base', BASES[seed]), ('single_recovery', SINGLES[seed])):
        name = f'{seed}/{label}'
        output = eval_path(name, 'val', checkpoint)
        evaluate(checkpoint, output, 'val', manifest=MANIFEST)
        VAL_EVALS[name] = output
    rows = json.loads((VAL_EVALS[f'{seed}/single_recovery'] / 'per_image.json').read_text())
    screen = validation_screen(rows)
    SINGLE_SCREENS.append(screen)
    audit = audit_checkpoint(SINGLES[seed], MANIFEST, SUITE_ROOT / f'diagnostics/single_recovery_{seed}.json')
    display(pd.DataFrame(audit['rows']))
    display(pd.DataFrame(paired_intervals(rows)))
    print('Validation development screen:', screen)
write_json(SUITE_ROOT / 'reports/single_screen.json', SINGLE_SCREENS)
if REQUIRE_RECOVERY_SCREEN and not all(s['passed'] for s in SINGLE_SCREENS):
    raise RuntimeError('Single expert did not improve validation enough. Run the final bundle cell and share the evidence.')


## 9. Shared initializer and matched routing experiments

Each MoE starts from the same learned encoder/single expert. Tiny independent RGB
projection perturbations break symmetry. Two full-coverage warm-up epochs observe
proposal benefit throughout the image; later epochs use the declared sparse budget.
The router's auxiliary loss is prevented from updating the reconstruction encoder.
Warm-up checkpoints cannot become the selected deployed checkpoint.

`sparse_error` predicts base error; `sparse_gain` predicts realized proposal benefit.
Both use identical architectures, data, losses, training duration, and inference
budget except the risk label. Unexecuted tiles are excluded from gain supervision.
This label remains an imperfect, changing estimate of recoverability, not an oracle.


In [ ]:
EXPERIMENTS = {
    'dense_gain': ('dense_transformer', 'gain'),
    'sparse_error': ('sparse_transformer', 'error'),
    'sparse_gain': ('sparse_transformer', 'gain'),
    'sparse_no_trust': ('sparse_no_trust', 'gain'),
    'sparse_uniform': ('sparse_uniform', 'gain'),
    'sparse_conv': ('sparse_conv', 'gain'),
    'sparse_adversarial': ('sparse_adversarial', 'gain'),
    'adaptive_k': ('adaptive_k', 'gain'),
}

def train_experiment(label):
    if not RUN.get(label, False):
        print('Skipped:', label)
        return
    profile, risk = EXPERIMENTS[label]
    for seed in SEEDS:
        config = new_config(seed, label, profile, initializer=SINGLES[seed], risk=risk)
        CONFIGS[f'{seed}/{label}'] = config
        MODELS[f'{seed}/{label}'] = launch(config)
    write_json(SUITE_ROOT / 'model_registry.json', {k: str(v.relative_to(SUITE_ROOT)) for k, v in MODELS.items()})


## 10. Dense MoE control


In [ ]:
train_experiment('dense_gain')


## 11. Sparse base-error router control


In [ ]:
train_experiment('sparse_error')


## 12. Sparse proposal-benefit router


In [ ]:
train_experiment('sparse_gain')


## 13. Remove trust


In [ ]:
train_experiment('sparse_no_trust')


## 14. Uniform regional coverage control


In [ ]:
train_experiment('sparse_uniform')


## 15. CNN router control (optional)


In [ ]:
train_experiment('sparse_conv')


## 16. Adversarial contribution (optional)


In [ ]:
train_experiment('sparse_adversarial')


## 17. Adaptive expert slots (optional)


In [ ]:
train_experiment('adaptive_k')


## 18. Legacy expert with the new curriculum (optional)

This control isolates the HR-conditioned expert architecture. It receives the
same single-expert loss and duration but has the old expert capacity and no HR
cues. It is trained from scratch using the shared base, so it does not inherit
an incompatible HR expert checkpoint. Report parameter differences explicitly.


In [ ]:
if RUN['legacy_expert_control']:
    for seed in SEEDS:
        config = new_config(seed, 'legacy_expert_control', hr=False)
        CONFIGS[f'{seed}/legacy_expert_control'] = config
        MODELS[f'{seed}/legacy_expert_control'] = launch(config)


## 19. Validation comparisons, routing, learning curves and measured runtime


In [ ]:
TIMINGS, HISTORIES = [], []
for name, checkpoint in MODELS.items():
    output = eval_path(name, 'val', checkpoint)
    evaluate(checkpoint, output, 'val', manifest=MANIFEST)
    VAL_EVALS[name] = output
    for row in benchmark(checkpoint, output / 'timing.json', manifest=MANIFEST):
        TIMINGS.append(dict(experiment=name, **row))
    for history in sorted(checkpoint.parent.glob('history/epoch_*.json')):
        row = json.loads(history.read_text())
        diagnostics = row.pop('first_batch_diagnostics', {})
        row.pop('weighted_objective', None)
        row.pop('first_batch_gradient_norms', None)
        HISTORIES.append(dict(experiment=name, **row, **diagnostics))
VAL_TABLES = summarize(VAL_EVALS, SUITE_ROOT / 'reports/validation')
display(VAL_TABLES['overall'][['experiment', 'count', *METRICS, 'psnr_delta_vs_base', 'correction_abs_mean']])
display(VAL_TABLES['expert_usage'])
TIMING_TABLE = pd.DataFrame(TIMINGS)
TIMING_TABLE.to_csv(SUITE_ROOT / 'reports/timing.csv', index=False)
display(TIMING_TABLE[['experiment', 'operation', 'parameters_total', 'parameters_used_on_this_frame',
                     'mean_ms', 'p95_ms', 'images_per_second', 'peak_allocated_mb']])
history = pd.DataFrame(HISTORIES)
history.to_csv(SUITE_ROOT / 'reports/training_history.csv', index=False)
if not history.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for name, group in history.groupby('experiment'):
        axes[0].plot(group.epoch, group.val_psnr, label=name)
        if 'correction_abs_mean' in group:
            axes[1].plot(group.epoch, group.correction_abs_mean, label=name)
    axes[0].set_title('Validation PSNR'); axes[1].set_title('First training batch: effective correction')
    axes[0].legend(fontsize=6); fig.tight_layout()
    fig.savefig(SUITE_ROOT / 'figures/learning.png', dpi=170); plt.show()
for seed in SEEDS:
    group = {k: v for k, v in VAL_EVALS.items() if k.startswith(f'{seed}/')}
    if f'{seed}/sparse_gain' in group:
        display(compare_controls(group, f'{seed}/sparse_gain', SUITE_ROOT / f'reports/validation/controls_{seed}.csv'))
write_json(SUITE_ROOT / 'model_registry.json', {k: str(v.relative_to(SUITE_ROOT)) for k, v in MODELS.items()})


## 20. Validation-only coverage ablation

Trained top-k stays fixed. Evaluate 25/50/75/100% coverage to measure a quality vs
computation curve. Final primary comparisons retain the trained coverage; these
curves are diagnostic ablations. A reduced fraction is not proof of lower latency.


In [ ]:
BUDGET_ROWS = []
for seed in SEEDS:
    name = f'{seed}/sparse_gain'
    if name not in MODELS:
        continue
    for fraction in (0.25, 0.5, 0.75, 1.0):
        output = eval_path(name, 'val', MODELS[name], f'coverage_{fraction}')
        scores = evaluate(MODELS[name], output, 'val', coverage=fraction,
                          manifest=MANIFEST, save_images=False)
        timing = benchmark(MODELS[name], output / 'timing.json', manifest=MANIFEST, coverage=fraction)[1]
        BUDGET_ROWS.append(dict(seed=seed, coverage=fraction, **scores,
                               mean_ms=timing['mean_ms'], images_per_second=timing['images_per_second']))
budget = pd.DataFrame(BUDGET_ROWS)
budget.to_csv(SUITE_ROOT / 'reports/validation_coverage.csv', index=False)
display(budget)


## 21. Freeze the test plan after reviewing validation

The proposed primary model is sparse_gain at the trained coverage and top-k.
Record whether it improves over both the base and the strong single expert;
failure is retained in the report. All enabled controls are tested. No checkpoint
or budget is chosen by looking at test scores. Prior repeated OLI2MSI testing means
a new geographically independent dataset is still needed for a strong final claim.


In [ ]:
plan_path = SUITE_ROOT / 'test_plan.json'
if plan_path.exists():
    TEST_PLAN = json.loads(plan_path.read_text())
    if TEST_PLAN['dataset_id'] != AUDIT['dataset_id']:
        raise ValueError('Test plan belongs to different data')
else:
    TEST_PLAN = dict(dataset_id=AUDIT['dataset_id'], protocol=DATA_CARD['protocol'],
                     primary='sparse_gain', coverage=REGION_FRACTION, top_k=TOP_K,
                     development_screens=SINGLE_SCREENS, checkpoints={})
    for name, source in MODELS.items():
        identity = digest_file(source)
        destination = SUITE_ROOT / 'test_models' / name / identity[:12] / 'best.pt'
        destination.parent.mkdir(parents=True, exist_ok=True)
        if not destination.exists():
            shutil.copy2(source, destination)
        if digest_file(destination) != identity:
            raise RuntimeError('Test checkpoint copy differs')
        TEST_PLAN['checkpoints'][name] = dict(path=str(destination.relative_to(SUITE_ROOT)), sha256=identity)
    write_json(plan_path, TEST_PLAN)
print(json.dumps(TEST_PLAN, indent=2))


## 22. Test all 100 official pairs using the frozen plan


In [ ]:
TEST_EVALS = {}
if RUN_TEST_EVALUATION:
    if FAST_DEV_RUN:
        raise RuntimeError('Smoke results must not be reported as the official test benchmark')
    for name, item in TEST_PLAN['checkpoints'].items():
        checkpoint = SUITE_ROOT / item['path']
        if digest_file(checkpoint) != item['sha256']:
            raise RuntimeError('Test snapshot changed: ' + name)
        output = SUITE_ROOT / 'evaluation/test' / name
        scores = evaluate(checkpoint, output, 'test', manifest=MANIFEST)
        if scores['count'] != 100:
            raise RuntimeError('Expected all 100 official test pairs')
        TEST_EVALS[name] = output
    TEST_TABLES = summarize(TEST_EVALS, SUITE_ROOT / 'reports/test')
    display(TEST_TABLES['overall'][['experiment', 'count', *METRICS, 'psnr_delta_vs_base', 'correction_abs_mean']])
    display(TEST_TABLES['paired_intervals'])
    for seed in SEEDS:
        group = {k: v for k, v in TEST_EVALS.items() if k.startswith(f'{seed}/')}
        if f'{seed}/sparse_gain' in group:
            display(compare_controls(group, f'{seed}/sparse_gain', SUITE_ROOT / f'reports/test/controls_{seed}.csv'))
else:
    print('Test is disabled. After validation, set RUN_TEST_EVALUATION=True and rerun this cell.')


## 23. Indexed outputs, signed corrections and routing maps

`show_result(index=25)` defaults to test. Use split='val' before opening test.
Images use the protocol range directly (gamma=1), with one shared scale and a
maximum of three columns. Correction plots use a shared symmetric color range;
they are signed correction amplitudes, not an artificially enhanced prediction.


In [ ]:
from geodiff_gan.experiments.trust_report import visualize_saved

def show_result(index=0, split='test', show_base=True, profiles=None, seed=42):
    profiles = profiles or ['single_recovery', 'sparse_error', 'sparse_gain']
    registry = TEST_EVALS if split == 'test' else VAL_EVALS
    choices = {name.split('/', 1)[1]: path for name, path in registry.items()
               if name.startswith(f'{seed}/') and name.split('/', 1)[1] in profiles}
    if not choices:
        raise ValueError('Run the corresponding evaluation cell first.')
    destination = SUITE_ROOT / f'figures/{split}_{seed}_{index:06d}.png'
    figures = visualize_saved(choices, MANIFEST, index=index, split=split, show_base=show_base,
                               output_path=destination, display_max=1.0, gamma=1.0)
    corrections = []
    for name, path in choices.items():
        with np.load(path / 'images' / f'{index:06d}.npz', allow_pickle=False) as values:
            corrections.append((name, (values['prediction'] - values['base']).mean(0)))
    limit = max(1e-5, max(float(np.quantile(np.abs(a), .99)) for _, a in corrections))
    columns = min(3, len(corrections))
    fig, axes = plt.subplots(int(np.ceil(len(corrections)/columns)), columns,
                             figsize=(5*columns, 4*int(np.ceil(len(corrections)/columns))), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, (name, array) in zip(axes.flat, corrections):
        picture = ax.imshow(array, cmap='RdBu_r', vmin=-limit, vmax=limit)
        ax.set_title(name + ': mean signed RGB correction')
    fig.colorbar(picture, ax=list(axes.flat), shrink=.7, label='Protocol intensity units')
    fig.savefig(destination.with_name(destination.stem + '_corrections.png'), dpi=170)
    plt.show()
    return figures

show_result(0, split='val')


## 24. Download the evidence and restart bundle

This cell can run even after a failed development screen. Download the ZIP and
save the executed notebook. Reports distinguish training diagnostics, validation,
test, failure, and measured runtime. Best/last checkpoints and code are included.
Full source tiles and prepared input caches are excluded. Six paired TIFF previews
are included for interpreting results without transferring the full dataset.


In [ ]:
import rasterio
from rasterio.transform import Affine

if 'BASES' in globals() and BASES:
    cfg = read_checkpoint_config(next(iter(BASES.values())))
    cfg['manifest'] = str(MANIFEST)
    split = 'test' if globals().get('TEST_EVALS') else 'val'
    data = dataset_for(cfg, split)
    folder = SUITE_ROOT / 'example_inputs'; folder.mkdir(exist_ok=True)
    for index in range(min(6, len(data))):
        sample = data[index]
        for key in ('lr', 'hr'):
            array = sample[key].numpy().astype('float32')
            # Benchmark crops have no verified georeferencing; do not invent a CRS.
            path = folder / f'{split}_{index:06d}_{key}.tif'
            with rasterio.open(path, 'w', driver='GTiff', width=array.shape[2], height=array.shape[1],
                               count=3, dtype='float32', compress='deflate', transform=Affine.identity()) as tif:
                tif.write(array)
                tif.update_tags(description='Protocol-normalized RGB example; no geographic CRS')
documentation = REPOSITORY_DIR / 'learning/trust_moe/07_residual_recovery_v2.md'
if documentation.exists():
    shutil.copy2(documentation, SUITE_ROOT / 'research_protocol.md')
notebook = REPOSITORY_DIR / 'kaggle/GeoDiff_TrustMoE_OLI2MSI_Residual_Recovery_3x.ipynb'
if notebook.exists():
    shutil.copy2(notebook, SUITE_ROOT / notebook.name)
archive = Path('/kaggle/working') / f'trust_recovery_results_{time.strftime("%Y%m%d_%H%M%S")}.zip'
bundle_results(SUITE_ROOT, archive, source_root=REPOSITORY_DIR, include_resume=True)
os.chdir('/kaggle/working')
print(f'Results and resume bundle: {archive} ({archive.stat().st_size / 2**20:.1f} MiB)')
display(FileLink(archive.name, result_html_prefix='Download results: '))
